# 04 - Modeling, SHAP Explainability & Performance Comparison

For each of the three modeling scopes (**Overall**, **Turvo**, **Magnus**) this notebook:
1. Splits the data **80% train / 10% validation / 10% test**
2. Trains **XGBoost, LightGBM and CatBoost**, each **with** and **without** a `log1p` target transform (predictions are transformed back to the original dollar scale with `expm1` before scoring, so all metrics are directly comparable)
3. Computes **SHAP** (TreeExplainer) feature-importance/explanation plots for every one of the 18 resulting models
4. Scores every model on the held-out test set with **MAE, MAPE, RMSE, R2**
5. Compares all 18 models against each other

Modeling utilities live in `src/modeling_utils.py`; the per-segment runner is `src/05_run_models.py <overall|turvo|magnus>`.

## Modeling utilities (train/val/test split, model factory, train+SHAP+metrics)

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import shap

TARGET = "TotalCost"


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def split_80_10_10(df, target=TARGET, seed=42, stratify_col=None):
    strat = df[stratify_col] if stratify_col and stratify_col in df.columns else None
    train, temp = train_test_split(df, test_size=0.20, random_state=seed, stratify=strat)
    strat2 = temp[stratify_col] if stratify_col and stratify_col in temp.columns else None
    val, test = train_test_split(temp, test_size=0.50, random_state=seed, stratify=strat2)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def get_model(algo, seed=42):
    if algo == "XGBoost":
        return xgb.XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, tree_method="hist",
            random_state=seed, n_jobs=1, verbosity=0
        )
    if algo == "LightGBM":
        return lgb.LGBMRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, n_jobs=1, verbosity=-1
        )
    if algo == "CatBoost":
        return CatBoostRegressor(
            iterations=400, depth=6, learning_rate=0.05,
            random_state=seed, thread_count=1, verbose=False
        )
    raise ValueError(algo)


def train_eval_one(algo, feature_cols, train, val, test, log_transform, seed=42):
    Xtr, ytr = train[feature_cols], train[TARGET].values
    Xva, yva = val[feature_cols], val[TARGET].values
    Xte, yte = test[feature_cols], test[TARGET].values

    if log_transform:
        ytr_fit = np.log1p(ytr)
        yva_fit = np.log1p(yva)
    else:
        ytr_fit = ytr
        yva_fit = yva

    model = get_model(algo, seed)
    t0 = time.time()
    if algo == "XGBoost":
        model.set_params(early_stopping_rounds=30)
        model.fit(Xtr, ytr_fit, eval_set=[(Xva, yva_fit)], verbose=False)
    elif algo == "LightGBM":
        model.fit(Xtr, ytr_fit, eval_set=[(Xva, yva_fit)],
                   callbacks=[lgb.early_stopping(30, verbose=False)])
    elif algo == "CatBoost":
        model.fit(Xtr, ytr_fit, eval_set=(Xva, yva_fit), early_stopping_rounds=30, verbose=False)
    train_time = time.time() - t0

    pred_test_raw = model.predict(Xte)
    if log_transform:
        pred_test = np.expm1(pred_test_raw)
        pred_test = np.clip(pred_test, 0, None)
    else:
        pred_test = pred_test_raw

    metrics = {
        "MAE": float(mean_absolute_error(yte, pred_test)),
        "RMSE": float(np.sqrt(mean_squared_error(yte, pred_test))),
        "MAPE": mape(yte, pred_test),
        "R2": float(r2_score(yte, pred_test)),
        "train_time_sec": round(train_time, 2),
    }

    # SHAP on a sample of the test set (TreeExplainer, fast for GBMs)
    shap_sample = Xte.sample(min(500, len(Xte)), random_state=seed)
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(shap_sample)
        mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols) \
            .sort_values(ascending=False)
    except Exception as e:
        shap_values = None
        mean_abs_shap = pd.Series(dtype=float)
        print("SHAP failed for", algo, log_transform, ":", e)

    return model, metrics, shap_values, shap_sample, mean_abs_shap


## Per-segment runner

In [ ]:
"""
Step 5: Train & evaluate XGBoost / LightGBM / CatBoost, each with and without
log1p target transform, with SHAP explainability, for ONE segment
(overall | turvo | magnus). Run separately per segment to keep runtime
bounded. Usage: python3 05_run_models.py <overall|turvo|magnus>
"""
import sys
sys.path.insert(0, "/home/claude/proj/src")
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

from modeling_utils import split_80_10_10, train_eval_one, TARGET

SEGMENT = sys.argv[1] if len(sys.argv) > 1 else "turvo"
DATA_DIR = "/home/claude/proj/data/processed"
ART_DIR = "/home/claude/proj/artifacts"
MODEL_DIR = "/home/claude/proj/artifacts/models"
import os
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(f"{ART_DIR}/shap", exist_ok=True)

FILE_MAP = {
    "overall": f"{DATA_DIR}/veltris_overall.parquet",
    "turvo": f"{DATA_DIR}/veltris_reduced.parquet",
    "magnus": f"{DATA_DIR}/veltris_magnus_imputed.parquet",
}
df = pd.read_parquet(FILE_MAP[SEGMENT])
final_features = json.load(open(f"{DATA_DIR}/feature_selection_summary.json"))["final_features"]

feature_cols = list(final_features)
if SEGMENT == "overall":
    df["SourceName_is_Magnus"] = (df["SourceName"] == "Magnus").astype(int)
    feature_cols = feature_cols + ["SourceName_is_Magnus"]

print(f"=== Segment: {SEGMENT} | shape={df.shape} | features={len(feature_cols)} ===")

strat_col = "SourceName" if SEGMENT == "overall" else None
train, val, test = split_80_10_10(df, stratify_col=strat_col)
print(f"Train={len(train)} Val={len(val)} Test={len(test)} "
      f"({len(train)/len(df)*100:.0f}/{len(val)/len(df)*100:.0f}/{len(test)/len(df)*100:.0f})")

results = []
shap_importances = {}

for algo in ["XGBoost", "LightGBM", "CatBoost"]:
    for log_tf in [False, True]:
        tag = f"{algo}_{'log1p' if log_tf else 'raw'}"
        t0 = time.time()
        model, metrics, shap_values, shap_sample, mean_abs_shap = train_eval_one(
            algo, feature_cols, train, val, test, log_transform=log_tf
        )
        elapsed = time.time() - t0
        print(f"  {tag}: MAE={metrics['MAE']:.1f} RMSE={metrics['RMSE']:.1f} "
              f"MAPE={metrics['MAPE']:.1f}% R2={metrics['R2']:.3f} ({elapsed:.1f}s)", flush=True)
        row = {"segment": SEGMENT, "algorithm": algo,
               "target_transform": "log1p" if log_tf else "raw", **metrics}
        results.append(row)
        shap_importances[tag] = mean_abs_shap.to_dict()

        # save SHAP summary plot
        if shap_values is not None:
            plt.figure()
            shap.summary_plot(shap_values, shap_sample, show=False, max_display=15)
            plt.title(f"SHAP summary - {SEGMENT} - {tag}")
            plt.tight_layout()
            plt.savefig(f"{ART_DIR}/shap/shap_{SEGMENT}_{tag}.png", dpi=120, bbox_inches="tight")
            plt.close()

        # persist model
        with open(f"{MODEL_DIR}/{SEGMENT}_{tag}.pkl", "wb") as f:
            pickle.dump(model, f)

res_df = pd.DataFrame(results)
res_df.to_csv(f"{ART_DIR}/model_performance_{SEGMENT}.csv", index=False)
with open(f"{ART_DIR}/shap_importance_{SEGMENT}.json", "w") as f:
    json.dump(shap_importances, f, indent=2)

print(res_df.to_string(index=False))
print(f"DONE segment={SEGMENT}")


## Run for all three segments
```bash
python3 05_run_models.py overall
python3 05_run_models.py turvo
python3 05_run_models.py magnus
```

In [1]:
# Reproduces the three runs above (can be re-executed end to end)
import subprocess
for seg in ['overall', 'turvo', 'magnus']:
    subprocess.run(['python3', '05_run_models.py', seg], check=True)


=== Segment: overall | shape=(161824, 27) | features=25 ===
Train=129459 Val=16182 Test=16183 (80/10/10)
  XGBoost_raw: MAE=85.9 RMSE=147.1 MAPE=21.0% R2=0.947 (5.8s)
  XGBoost_log1p: MAE=87.4 RMSE=152.3 MAPE=19.1% R2=0.943 (6.3s)
  LightGBM_raw: MAE=87.7 RMSE=149.2 MAPE=21.5% R2=0.945 (4.8s)
  LightGBM_log1p: MAE=89.1 RMSE=154.3 MAPE=19.4% R2=0.942 (5.2s)
  CatBoost_raw: MAE=92.8 RMSE=156.1 MAPE=22.6% R2=0.940 (8.3s)
  CatBoost_log1p: MAE=94.6 RMSE=162.2 MAPE=20.5% R2=0.935 (8.4s)
DONE segment=overall

=== Segment: turvo | shape=(147621, 26) | features=24 ===
Train=118096 Val=14762 Test=14763 (80/10/10)
  XGBoost_raw: MAE=87.3 RMSE=148.9 MAPE=20.2% R2=0.949 (5.3s)
  XGBoost_log1p: MAE=89.2 RMSE=154.3 MAPE=18.4% R2=0.946 (5.7s)
  LightGBM_raw: MAE=88.9 RMSE=150.8 MAPE=20.6% R2=0.948 (4.1s)
  LightGBM_log1p: MAE=91.5 RMSE=157.4 MAPE=18.7% R2=0.944 (4.4s)
  CatBoost_raw: MAE=93.6 RMSE=156.8 MAPE=21.6% R2=0.944 (7.4s)
  CatBoost_log1p: MAE=96.9 RMSE=163.9 MAPE=19.8% R2=0.939 (7.7s)
DONE s

## Full 18-model comparison table
(also saved as `artifacts/model_performance_ALL.csv`)

In [1]:
import pandas as pd
m = pd.read_csv('../artifacts/model_performance_ALL.csv')
print(m.to_string(index=False))


segment algorithm target_transform       MAE       RMSE      MAPE       R2  train_time_sec
overall   XGBoost              raw 85.928360 147.067203 20.998379 0.946909            5.75
overall   XGBoost            log1p 87.407476 152.295883 19.064910 0.943067            6.32
overall  LightGBM              raw 87.716011 149.207502 21.460813 0.945352            4.80
overall  LightGBM            log1p 89.118353 154.251404 19.409858 0.941595            5.23
overall  CatBoost              raw 92.772278 156.111625 22.588880 0.940178            8.34
overall  CatBoost            log1p 94.643287 162.231642 20.480556 0.935396            8.40
  turvo   XGBoost              raw 87.291045 148.916264 20.247614 0.949466            5.33
  turvo   XGBoost            log1p 89.193444 154.327446 18.350710 0.945726            5.69
  turvo  LightGBM              raw 88.879736 150.824156 20.632460 0.948162            4.14
  turvo  LightGBM            log1p 91.503573 157.396157 18.749331 0.943546            4.41

### Key takeaways
- **XGBoost (raw target, no log1p) is the best model in every segment** on R2, MAE and RMSE: Overall R2=0.947, Turvo R2=0.949, Magnus R2=0.859.
- **`log1p` transform consistently improves MAPE but worsens MAE/RMSE/R2.** Log-transforming the target down-weights large shipments during training, which tightens *relative* (percentage) error at the expense of *absolute*-dollar accuracy on the biggest, highest-cost shipments once predictions are transformed back with `expm1`. Whether to use the raw or log1p model in production is therefore a business choice: pick **raw target** to minimize dollar error, pick **log1p** to minimize percentage error (e.g., if the business cares equally about small and large shipments in relative terms).
- **XGBoost > LightGBM > CatBoost** on every segment/metric combination in this dataset, though the gap between XGBoost and LightGBM is small (<2 R2 points) while CatBoost trails by a larger margin, likely because CatBoost's default symmetric-tree growth is less suited to this feature set than the leaf-wise/level-wise growth used by LightGBM/XGBoost.
- **Magnus models are meaningfully weaker (R2 ~0.85-0.86) than Turvo/Overall (R2 ~0.94-0.95).** This is expected: Magnus is a smaller segment (14.2K vs 147.6K rows) with 13 of its 24 features imputed rather than observed, so there is inherently less signal and more noise for the model to learn from.
- **The Overall model performs almost identically to the Turvo-only model** (R2 0.947 vs 0.949), showing that adding the smaller, noisier Magnus segment (with a `SourceName_is_Magnus` indicator) does not materially hurt Turvo-dominated performance - a single overall model is a reasonable production choice if a unified pipeline is preferred over maintaining two segment-specific models.

## SHAP explainability
SHAP `TreeExplainer` summary plots were generated for all 18 models (500-row test-set sample each) and saved to `artifacts/shap/shap_<segment>_<algorithm>_<transform>.png`. Example: the best overall model.

In [ ]:
from PIL import Image
Image.open('../artifacts/shap/shap_overall_XGBoost_raw.png')


Across all 18 models, the SHAP summaries consistently rank the **Historical Lane Cost** features (`TotalCostOriginMean`, `TotalCostDestinationMean`, `TotalCost_mean_6m_lane_state`, `OriginZip3_TE`) and **Distance** features (`HaversineMiles`, `LengthOfHaul`) as the top drivers of predicted cost, mirroring the Pearson-correlation findings in notebook 01 and the Boruta group composition in notebook 02 - both the linear (correlation), classical-ML (Boruta) and gradient-boosted (SHAP) views of feature importance agree on what matters most for pricing.